# DataSphere ML — polyglot predictors (Colab demo)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harshini-Suresha/datasphere-ml/blob/main/notebooks/datasphere_ml_demo.ipynb)
![Website](https://img.shields.io/badge/website-live-brightgreen)](https://harshini-suresha.github.io/datasphere-ml/) ![GitHub](https://img.shields.io/badge/github-repo-blue)](https://github.com/Harshini-Suresha/datasphere-ml)

**Live links:** [Website (github.io)](https://harshini-suresha.github.io/datasphere-ml/) · [GitHub repo](https://github.com/Harshini-Suresha/datasphere-ml) · [Streamlit app](https://datasphere-ml-i9nv6eofsxkupnbfsdnqrx.streamlit.app/) · [Download this notebook (.ipynb)](https://raw.githubusercontent.com/Harshini-Suresha/datasphere-ml/main/notebooks/datasphere_ml_demo.ipynb)

One synthetic e-commerce dataset (seed `42`) spanning MySQL-style orders, Mongo-style catalogue/reviews and Neo4j-style graph centrality. Trains 3 sklearn RandomForest models:
1. **High-value customer** classifier (top-20% spenders)
2. **Order amount** regressor
3. **Review rating** classifier (1–5 stars)

> Runs in Colab in ~1–2 min with **no extra installs** (sklearn/pandas/matplotlib are pre-installed). For the full RF+XGBoost+LightGBM+SHAP run, clone the repo and run `python train_models.py --fast`.


## 0 · Setup — run this first in Colab

No installs needed. Just check versions.

In [ ]:
import sklearn, pandas, numpy, matplotlib
print('sklearn', sklearn.__version__)
print('pandas', pandas.__version__)
print('numpy', numpy.__version__)
# Uncomment if you want the full repo (XGBoost/LightGBM/SHAP + Streamlit app):
# !pip install -q xgboost lightgbm shap streamlit plotly
# !git clone https://github.com/Harshini-Suresha/datasphere-ml.git


## 1 · Generate one synthetic dataset (seed 42)

Same generator logic as `train_models.py`, smaller sizes for Colab speed.

In [ ]:
import numpy as np
import pandas as pd

SEED = 42
CITIES = ['London', 'Bengaluru', 'Pune', 'Berlin', 'Austin', 'Lagos']
CITY_P = [0.28, 0.22, 0.15, 0.14, 0.12, 0.09]
CITY_PREM = {'London': 14, 'Austin': 10, 'Berlin': 8, 'Bengaluru': -2, 'Pune': -4, 'Lagos': -6}
CATS = ['Shoes', 'Electronics', 'Home', 'Books', 'Beauty', 'Sports']

def gen_customers(n=3000, seed=SEED):
    rng = np.random.default_rng(seed)
    degree = np.ones(n)
    for i in range(1, n):
        w = degree[:i] + 1.0
        p = w / w.sum()
        for _ in range(2 if i > 3 else 1):
            j = rng.choice(i, p=p)
            degree[i] += 1; degree[j] += 1
    pagerank = degree / degree.sum() * n
    city = rng.choice(CITIES, size=n, p=CITY_P)
    prem = np.array([CITY_PREM[c] for c in city])
    tenure = np.clip(rng.gamma(5.0, 90.0, n).astype(int), 30, 900)
    propensity = rng.normal(0, 1, n)
    activity = rng.gamma(2.0, 1.0, n) * (0.5 + pagerank) + 0.4 * propensity
    n_orders = np.clip((rng.power(1.6, n) * 34 * (0.4 + activity / np.mean(activity))
                        + (propensity > 1).astype(int) * 4).astype(int), 1, 120)
    avg_amount = np.clip(np.round(22 + prem + rng.gamma(2.2, 16, n)
                                    + (pagerank - 1) * 5 + propensity * 7, 2), 5, 400)
    total = np.round(n_orders * avg_amount * rng.lognormal(0, 0.12, n), 2)
    recency = np.clip((rng.exponential(45, n) / (0.5 + activity / 3)).astype(int), 0, 400)
    n_cat = np.clip(rng.integers(1, 6, n) + (activity > np.mean(activity)).astype(int), 1, 6)
    n_rev = (rng.poisson(2.5, n) + (activity > 2).astype(int) * 2).clip(0, 30)
    fav_idx = (rng.integers(0, 6, n) + (city == 'London') * 1 + (city == 'Bengaluru') * 3) % 6
    leniency = np.round(rng.normal(0, 0.6, n), 3)
    df = pd.DataFrame({'customer_id': np.arange(1, n+1), 'city': city, 'tenure_days': tenure,
                       'degree': degree.astype(int), 'pagerank': np.round(pagerank, 4),
                       'n_orders': n_orders, 'avg_amount': avg_amount, 'recency_days': recency,
                       'n_categories': n_cat, 'n_reviews': n_rev,
                       'fav_category': [CATS[i] for i in fav_idx], 'leniency': leniency,
                       'total_spent': total})
    score = np.log(total) + rng.normal(0, 0.30, n)
    df['high_value'] = (score >= np.quantile(score, 0.8)).astype(int)
    return df

def gen_orders(cust, n_orders=12000, seed=SEED+1):
    rng = np.random.default_rng(seed)
    cid = rng.choice(cust['customer_id'].values, size=n_orders,
                     p=(cust['n_orders'] / cust['n_orders'].sum()).values)
    c = cust.set_index('customer_id').loc[cid].reset_index(drop=True)
    month = rng.integers(1, 13, n_orders); dow = rng.integers(0, 7, n_orders)
    cat = [CATS[i] for i in rng.integers(0, 6, n_orders)]
    cat_eff = {'Shoes': 8, 'Electronics': 55, 'Home': 12, 'Books': -8, 'Beauty': 2, 'Sports': 15}
    weekend = (dow >= 5).astype(int)
    wknd_eff = np.array([8 if x in ('Shoes','Beauty','Sports') else (-5 if x == 'Electronics' else 0) for x in cat]) * weekend
    seas_eff = (month == 12) * 25 + (month == 11) * 10 - (month == 1) * 8
    base = (c['avg_amount'].values * 0.50 + np.minimum(c['tenure_days'].values, 600) * 0.015
            + np.array([cat_eff[x] for x in cat]) + wknd_eff + seas_eff)
    amount = base * rng.lognormal(0, 0.28, n_orders)
    return pd.DataFrame({'tenure_days': c['tenure_days'].values, 'past_avg': c['avg_amount'].values,
                         'degree': c['degree'].values, 'pagerank': c['pagerank'].values,
                         'recency_days': c['recency_days'].values, 'month': month,
                         'day_of_week': dow, 'category': cat,
                         'amount': np.round(np.clip(amount, 5, 900), 2)})

def gen_reviews(cust, n_rev=8000, seed=SEED+2):
    rng = np.random.default_rng(seed)
    w = (cust['n_reviews'] + 0.5).values
    cid = rng.choice(cust['customer_id'].values, size=n_rev, p=w / w.sum())
    c = cust.set_index('customer_id').loc[cid].reset_index(drop=True)
    price = np.round(10 + rng.power(2, n_rev) * 290, 2)
    cat = [CATS[i] for i in rng.integers(0, 6, n_rev)]
    reviewer_avg = np.clip(3.8 + c['leniency'].values * 0.7 + rng.normal(0, 0.15, n_rev), 1, 5)
    quality = (np.array([0.3 if x == 'Books' else (-0.2 if x == 'Electronics' else 0.0) for x in cat])
               - (price > 200) * 0.5 + rng.normal(0, 0.5, n_rev))
    latent = 4.0 + quality * 0.7 + (reviewer_avg - 3.8) * 0.9
    stars = np.clip(np.round(latent + rng.normal(0, 0.5, n_rev)), 1, 5).astype(int)
    return pd.DataFrame({'price': price, 'category': cat,
                         'reviewer_n_reviews': c['n_reviews'].values,
                         'reviewer_avg_stars': np.round(reviewer_avg, 2),
                         'reviewer_pagerank': c['pagerank'].values,
                         'reviewer_recency': c['recency_days'].values, 'stars': stars})

cust = gen_customers(); ords = gen_orders(cust); revs = gen_reviews(cust)
print(cust.shape, ords.shape, revs.shape)
print('high-value rate:', round(cust.high_value.mean(), 3))
cust.head(3)


## 2 · Train: high-value customer classifier

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

def enc(df, cat_cols):
    return ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)], remainder='passthrough')

feats_c = ['city','tenure_days','degree','pagerank','n_orders','avg_amount','recency_days','n_categories','n_reviews','fav_category']
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(cust[feats_c], cust['high_value'], test_size=0.2,
                                        stratify=cust['high_value'], random_state=42)
pipe_c = Pipeline([('enc', enc(Xc_tr, ['city','fav_category'])),
                   ('m', RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))])
pipe_c.fit(Xc_tr, yc_tr)
proba = pipe_c.predict_proba(Xc_te)[:, 1]; pred = (proba > 0.5).astype(int)
print(f"ROC-AUC {roc_auc_score(yc_te, proba):.4f} | F1 {f1_score(yc_te, pred):.4f} | acc {accuracy_score(yc_te, pred):.4f}")


## 3 · Train: order-amount regressor + rating classifier

In [ ]:
feats_o = ['tenure_days','past_avg','degree','pagerank','recency_days','month','day_of_week','category']
Xo_tr, Xo_te, yo_tr, yo_te = train_test_split(ords[feats_o], ords['amount'], test_size=0.2, random_state=42)
pipe_o = Pipeline([('enc', enc(Xo_tr, ['category'])),
                   ('m', RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42))])
pipe_o.fit(Xo_tr, yo_tr)
pr = pipe_o.predict(Xo_te)
print(f"R2 {r2_score(yo_te, pr):.4f} | MAE {mean_absolute_error(yo_te, pr):.2f} | RMSE {float(((yo_te-pr)**2).mean()**0.5):.2f}")

feats_r = ['price','category','reviewer_n_reviews','reviewer_avg_stars','reviewer_pagerank','reviewer_recency']
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(revs[feats_r], revs['stars']-1, test_size=0.2,
                                        stratify=revs['stars'], random_state=42)
pipe_r = Pipeline([('enc', enc(Xr_tr, ['category'])),
                   ('m', RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))])
pipe_r.fit(Xr_tr, yr_tr)
pr = pipe_r.predict(Xr_te)
print(f"rating acc {accuracy_score(yr_te, pr):.4f} | F1-macro {f1_score(yr_te, pr, average='macro'):.4f}")


## 4 · Plots: calibration, feature importance, distributions

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
# (a) calibration: mean predicted vs observed per decile
proba_c = pipe_c.predict_proba(Xc_te)[:, 1]
edges = np.linspace(0, 1, 11); xs, ys = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (proba_c > lo) & (proba_c <= hi) if lo > 0 else (proba_c >= lo) & (proba_c <= hi)
    if m.sum() > 0:
        xs.append(proba_c[m].mean()); ys.append(np.asarray(yc_te)[m].mean())
ax[0].plot([0,1],[0,1],'--',color='grey'); ax[0].scatter(xs, ys); ax[0].set_title('Calibration (customer)')
ax[0].set_xlabel('mean predicted'); ax[0].set_ylabel('observed rate')
# (b) top importances, amount model
names = list(pipe_o.named_steps['enc'].get_feature_names_out())
imp = pipe_o.named_steps['m'].feature_importances_
o = np.argsort(imp)[::-1][:8]
ax[1].barh([names[i] for i in o][::-1], imp[o][::-1]); ax[1].set_title('Top features (amount)')
# (c) order amount distribution
ax[2].hist(ords['amount'], bins=40); ax[2].set_title('Order amount (£)'); ax[2].set_xlabel('£')
plt.tight_layout(); plt.show()


## 5 · Try one prediction (edit me!)

Same features the Streamlit app and the `index.html` calculators use.

In [ ]:
demo_customer = pd.DataFrame([{'city':'London','tenure_days':400,'degree':12,'pagerank':2.1,'n_orders':25,
                                     'avg_amount':85.0,'recency_days':9,'n_categories':4,
                                     'n_reviews':6,'fav_category':'Shoes'}])
print('P(high-value) =', round(float(pipe_c.predict_proba(demo_customer)[0,1]), 3))

demo_order = pd.DataFrame([{'tenure_days':400,'past_avg':85.0,'degree':12,'pagerank':2.1,
                               'recency_days':9,'month':12,'day_of_week':5,'category':'Electronics'}])
print('Predicted order £ =', round(float(pipe_o.predict(demo_order)[0]), 2))

demo_review = pd.DataFrame([{'price':89.0,'category':'Shoes','reviewer_n_reviews':4,
                                'reviewer_avg_stars':4.2,'reviewer_pagerank':1.5,'reviewer_recency':30}])
print('Predicted stars =', int(pipe_r.predict(demo_review)[0]) + 1)


## Next steps

- **Website:** https://harshini-suresha.github.io/datasphere-ml/ (interactive SQL lab, graph views, in-page ML calculators)
- **Full training** (RF + XGBoost + LightGBM + SHAP + calibration JSONs):
  ```bash
  git clone https://github.com/Harshini-Suresha/datasphere-ml.git
  cd datasphere-ml
  pip install -r requirements.txt
  python train_models.py --fast   # or full: python train_models.py
  python export_js.py             # refresh in-page calculators (ml_models.js)
  streamlit run streamlit_app/ml_app.py
  ```
- **Streamlit app:** https://datasphere-ml-i9nv6eofsxkupnbfsdnqrx.streamlit.app/
